# Kaggle ML Challenge 03 — Ensemble Wars: Aging Electronic Nose

**Objective:** Predict the gas class (`label`) for each test observation using an ensemble of independently trained traditional machine-learning models.

**Models used**
- RBF Support Vector Machine (SVM)
- Extra Trees Classifier
- Weighted soft voting: 90% SVM + 10% Extra Trees

**Evaluation:** Chronological validation on batches 6 and 7 using Macro-F1. The final models are trained on all available training rows.

> Important: Validation performance is not a guarantee of hidden-test Kaggle performance. Do not assume that a higher local score will produce a higher leaderboard score.


## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score, classification_report

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load the datasets

In [2]:
# Keep train.csv, test.csv and sample_submission.csv in the same folder as this notebook.

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print("Training dataset:", train.shape)
print("Testing dataset:", test.shape)
print("Sample submission:", sample_submission.shape)

display(train.head())

# Basic checks
assert "label" in train.columns
assert "Id" in train.columns and "Id" in test.columns
assert len(test) == len(sample_submission)
assert train["label"].notna().all()

print("\\nTraining batches:")
print(train["batch"].value_counts().sort_index())

print("\\nClass distribution:")
print(train["label"].value_counts().sort_index())

Training dataset: (9546, 131)
Testing dataset: (4364, 130)
Sample submission: (4364, 2)


,Id,batch,label,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,...,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128
0,11813,7,6,18426.1433,8.012311,4.637280,6.854782,10.755495,-2.700655,-3.956627,...,-7.649273,-15.691021,20123.4307,4.265258,6.735639,11.602448,16.781786,-4.716063,-7.207139,-15.563540
1,875,7,4,12881.6084,4.999426,2.700797,3.776042,7.609812,-1.719600,-2.495682,...,-8.172268,-13.557608,17969.0107,4.448080,5.905540,10.604961,13.701524,-4.375042,-7.002332,-12.966454
2,428,3,3,21118.7700,5.584744,5.340035,7.677719,8.328934,-3.701702,-4.710054,...,-7.272255,-9.995701,15885.3477,2.842874,5.417050,8.220536,9.267007,-3.889974,-6.030040,-8.723799
3,3249,7,2,19086.3298,7.184287,4.100390,6.309003,10.159742,-3.291821,-4.671168,...,-17.518615,-34.194218,37958.7178,12.322653,19.864813,51.314521,63.511047,-9.081936,-14.357400,-24.870645
4,2146,6,6,54701.7856,7.965081,16.761078,27.019008,36.757151,-9.944200,-13.916737,...,-5.870580,-11.506023,16154.8311,4.488141,4.503564,7.284964,10.996612,-3.259098,-5.218725,-11.085125


\nTraining batches:
batch
1     445
2    1244
3    1586
4     161
5     197
6    2300
7    3613
Name: count, dtype: int64
\nClass distribution:
label
1    1874
2    2241
3     901
4    1228
5    2188
6    1114
Name: count, dtype: int64


## 3. Prepare features and chronological validation splits

In [3]:
feature_cols = [col for col in train.columns if col.startswith("feature_")]

X = train[feature_cols]
y = train["label"]

X_test = test[feature_cols]

print("Number of features:", len(feature_cols))
print("Training feature shape:", X.shape)
print("Test feature shape:", X_test.shape)
print("Missing training feature values:", X.isna().sum().sum())
print("Missing test feature values:", X_test.isna().sum().sum())

# Chronological validation:
# For batch 6, train only on batches earlier than 6.
# For batch 7, train only on batches earlier than 7.


Number of features: 128
Training feature shape: (9546, 128)
Test feature shape: (4364, 128)
Missing training feature values: 0
Missing test feature values: 0


## 4. Define the two independent models

In [4]:
# Model 1: RBF Support Vector Machine
# StandardScaler is used because SVM is sensitive to feature scales.

svm_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=30,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=42
    ))
])

# Model 2: Extra Trees
# Extra Trees is a tree-based model and does not require feature scaling.

extra_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", ExtraTreesClassifier(
        n_estimators=600,
        max_features="sqrt",
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ))
])

print("Both model pipelines are defined.")

Both model pipelines are defined.


## 5. Chronological validation on batches 6 and 7

Each validation batch is predicted using only earlier batches. This better reflects the chronological structure of the challenge than a random train-validation split.

In [5]:
validation_results = []
saved_validation_models = {}

for validation_batch in [6, 7]:
    train_mask = train["batch"] < validation_batch
    valid_mask = train["batch"] == validation_batch

    X_tr, y_tr = X.loc[train_mask], y.loc[train_mask]
    X_v, y_v = X.loc[valid_mask], y.loc[valid_mask]

    svm_fold = clone(svm_model)
    extra_fold = clone(extra_model)

    svm_fold.fit(X_tr, y_tr)
    extra_fold.fit(X_tr, y_tr)

    svm_pred = svm_fold.predict(X_v)
    extra_pred = extra_fold.predict(X_v)

    svm_score = f1_score(y_v, svm_pred, average="macro")
    extra_score = f1_score(y_v, extra_pred, average="macro")

    # Align Extra Trees probability columns to the SVM class order.
    svm_classes = svm_fold.named_steps["classifier"].classes_
    extra_classes = extra_fold.named_steps["classifier"].classes_

    svm_probs = svm_fold.predict_proba(X_v)
    extra_probs = extra_fold.predict_proba(X_v)
    extra_probs_aligned = np.column_stack([
        extra_probs[:, list(extra_classes).index(cls)]
        for cls in svm_classes
    ])

    # Weighted soft voting: 90% SVM + 10% Extra Trees.
    ensemble_probs = 0.90 * svm_probs + 0.10 * extra_probs_aligned
    ensemble_pred = svm_classes[np.argmax(ensemble_probs, axis=1)]
    ensemble_score = f1_score(y_v, ensemble_pred, average="macro")

    validation_results.append({
        "Validation Batch": validation_batch,
        "SVM Macro-F1": svm_score,
        "Extra Trees Macro-F1": extra_score,
        "Ensemble Macro-F1 (90:10)": ensemble_score
    })

    saved_validation_models[validation_batch] = {
        "svm": svm_fold,
        "extra": extra_fold
    }

validation_results_df = pd.DataFrame(validation_results)
display(validation_results_df)

print("\\nAverage ensemble Macro-F1 across these two batches:",
      round(validation_results_df["Ensemble Macro-F1 (90:10)"].mean(), 5))

,Validation Batch,SVM Macro-F1,Extra Trees Macro-F1,Ensemble Macro-F1 (90:10)
0,6,0.637814,0.645083,0.640686
1,7,0.878669,0.780822,0.887606


\nAverage ensemble Macro-F1 across these two batches: 0.76415


## 6. Train the final models on all training data

After chronological validation, refit both independent models using all labelled training rows (batches 1–7).

In [6]:
final_svm = clone(svm_model)
final_extra = clone(extra_model)

final_svm.fit(X, y)
final_extra.fit(X, y)

print("Final SVM and Extra Trees models trained on all training data.")

Final SVM and Extra Trees models trained on all training data.


## 7. Generate test predictions using weighted soft voting

In [7]:
svm_test_probs = final_svm.predict_proba(X_test)
extra_test_probs = final_extra.predict_proba(X_test)

# Align probability columns before combining them.
svm_classes = final_svm.named_steps["classifier"].classes_
extra_classes = final_extra.named_steps["classifier"].classes_

extra_test_probs_aligned = np.column_stack([
    extra_test_probs[:, list(extra_classes).index(cls)]
    for cls in svm_classes
])

# Weighted soft voting: 90% SVM + 10% Extra Trees.
final_probs = 0.90 * svm_test_probs + 0.10 * extra_test_probs_aligned
final_predictions = svm_classes[np.argmax(final_probs, axis=1)]

print("Predictions generated:", len(final_predictions))
print("Predicted classes:", np.unique(final_predictions))

Predictions generated: 4364
Predicted classes: [1 2 3 4 5 6]


## 8. Create the Kaggle submission file

In [8]:
# Create predictions indexed by test Id.
predictions_by_id = pd.Series(
    final_predictions,
    index=test["Id"]
)

# Preserve the exact row order from sample_submission.csv.
submission = sample_submission[["Id"]].copy()
submission["label"] = submission["Id"].map(predictions_by_id)

# Check for unmatched IDs before converting labels to integers.
assert submission["label"].notna().all(), "Some sample submission IDs have no prediction."
submission["label"] = submission["label"].astype(int)

# Save a new file; do not overwrite earlier submissions.
submission.to_csv("submission_ensemble_improved.csv", index=False)

print("Saved: submission_ensemble_improved.csv")
display(submission.head())
print("Submission shape:", submission.shape)
print("Missing labels:", submission["label"].isna().sum())
print("\\nPredicted class distribution:")
print(submission["label"].value_counts().sort_index())

Saved: submission_ensemble_improved.csv


,Id,label
0,2555,4
1,6408,5
2,2345,4
3,347,6
4,7030,6


Submission shape: (4364, 2)
Missing labels: 0
\nPredicted class distribution:
label
1    1023
2     624
3     619
4     694
5     501
6     903
Name: count, dtype: int64


## 9. Final submission checks

In [9]:
assert list(submission.columns) == ["Id", "label"]
assert len(submission) == len(test) == len(sample_submission)
assert submission["Id"].is_unique
assert submission["label"].notna().all()
assert submission["label"].between(1, 6).all()
assert submission["Id"].equals(sample_submission["Id"])

print("ALL SUBMISSION CHECKS PASSED!")
print("Final file: submission_ensemble_improved.csv")

ALL SUBMISSION CHECKS PASSED!
Final file: submission_ensemble_improved.csv


## 10. Conclusion

- Two independently trained traditional ML models are used: SVM and Extra Trees.
- Their predicted probabilities are combined using weighted soft voting.
- Chronological validation is performed on batches 6 and 7.
- The final models are trained on all labelled training rows.
- The submission file contains the required `Id` and `label` columns.

**Caution:** The validation batches are only a limited estimate of performance on future batches. The hidden test score can differ.